In [1]:
import os
import json
import numpy as np
import pandas as pd

## Specity the Inputs

You need to speficy the following path of the data for extraction

1. "prescient_file_path", the path of the Prescient output folder.
2. "thermal_detail_path", the path of the "thermal_detail.csv" in the Prescient output folder.
3. "renew_detail_path", the path of the "renewables_detail.csv" in the Prescient output folder.
4. "bus_detail_path", the path of the "bus_detail.csv" in the Prescient output folder.
5. "gen_dict", the path of json file that contains the information of the generators in your dataset. Run "Test_utils.ipynb" to get the gen_dict.json.

In [2]:
# specify your path of prescient file here
# Attention: change the path before running the cell
this_file_path = os.getcwd()
prescient_file_path = os.path.join(this_file_path, "..", "Data", "base_pcm_simulation_new_env")

# the bus_detail and thermal_detail csv
thermal_detail_path = os.path.join(prescient_file_path, "thermal_detail.csv")
renew_detail_path = os.path.join(prescient_file_path, "renewables_detail.csv")
bus_detail_path = os.path.join(prescient_file_path, "bus_detail.csv")

# the json file is the one used to store the information of the generators
json_path = os.path.join("..", "Data", "gen_dict.json")

## Check the Inputs

In [3]:
# Read the generator information from my files
with open(json_path, "rb") as f:
    gen_dict = json.load(f)
gen_dict["fossil"][gen_name].keys()

NameError: name 'gen_name' is not defined

In [ ]:
# check the if the generator is in the bus

def check_generator_and_bus(bus_name, gen_name):
    # check which type generator that the generator belongs to (renewable or fossil)
    if gen_name in gen_dict["fossil"].keys():
        gen_type = "fossil"
    elif gen_name in gen_dict["renew"].keys():
        gen_type = "renew"
    else:
        raise ValueError("The generator name is not vaild")
    
    # check the if the generator is in the bus provided.
    bus_name_ = gen_dict[gen_type][gen_name]["bus_name"]
    if bus_name == bus_name_:
        print(f"The generator {gen_name} is a {gen_type} generator. \nThe generator {gen_name} is in the bus {bus_name}.")
    
    else:
        raise ValueError(f"The generator {gen_name} should be located at bus {bus_name_}, \n but bus {bus_name} is provided.")
    
    return gen_type

In [ ]:
# You can also loop over all the generators and buses.

# For information of the bus and generator names, please visit 
# https://github.com/GridMod/RTS-GMLC/tree/master/RTS_Data/SourceData
bus_name = "Abel"
gen_name = "101_CT_1"

# run this cell to make sure your are reading the correct bus and generator information.
gen_type = check_generator_and_bus(bus_name, gen_name)

The generator 101_CT_1 is a fossil generator. 
The generator 101_CT_1 is in the bus Abel.


# Fossil

## Read LMP and Dispatch Results

In [ ]:
# read Prescient to pandas dataframe
def _prescient_output_to_df(file_name):
    '''Helper for loading data from Prescient output csv.
        Combines Datetimes into single column.
    '''
    df = pd.read_csv(file_name)
    df['Datetime'] = \
        pd.to_datetime(df['Date']) + \
        pd.to_timedelta(df['Hour'], 'hour') + \
        pd.to_timedelta(df['Minute'], 'minute')
    df.drop(columns=['Date','Hour','Minute'], inplace=True)
    # put 'Datetime' in front
    cols = df.columns.tolist()
    cols = cols[-1:]+cols[:-1]
    
    return df[cols]

In [ ]:
# read the lmp at the bus and put it in a csv
def make_lmp_csv(lmp_path, bus_details_path, bus_name):
    """
    This function reads all the LMP at the bus and put it into a dataframe.
    You can loop over all the buses and get all results in one csv.
    
    Args:
        lmp_path: str (path), the path for the csv you want to save the lmp.
        bus_details_path: str (path), the path of bus_detail.csv in the prescient results.
        bus_name: str, the name of the bus.
    """
    bdf = _prescient_output_to_df(bus_details_path)
    bdf = bdf[bdf["Bus"] == bus_name][["Datetime","LMP","LMP DA"]]
    bdf.set_index("Datetime", inplace=True)

    if lmp_path == None:
        print("Empty path, make a new df")
        bdf = bdf.rename(columns={'LMP': f'{bus_name}_LMP', "LMP DA": f'{bus_name}_LMP DA'})
        lmp_df = bdf
        lmp_df.to_csv("Bus_LMP.csv")
    else:
        print(f"Extracting LMP for {bus_name}")
        bdf = bdf.rename(columns={"LMP": f"{bus_name}_LMP", "LMP DA": f"{bus_name}_LMP DA"})
        lmp_df = pd.read_csv(lmp_path).set_index("Datetime")
        # check if the bus has already been read
        if (f"{bus_name}_LMP" in lmp_df.columns) or (f"{bus_name}_LMP_DA" in lmp_df.columns):
            print(f"{bus_name} LMP already exists.")
        else:
            bdf_aligned = bdf.reindex(lmp_df.index)
            lmp_df_merge = pd.concat([lmp_df, bdf_aligned], axis=1)
            lmp_df_merge.to_csv(lmp_path)
    
    return

In [ ]:
# read the dispatch at the bus and put it in a csv
def make_dispatch_csv(dispatch_path, gen_details_path, gen_name, gen_type=gen_type, other_info=None):
    """
    This function reads all the dispatch result of the generator and put it into a dataframe.
    You can loop over all the buses and get all results in one csv.
    
    Args:
        dispatch_path: str (path), the path for the csv you want to save the dispatch results.
        gen_details_path: str (path), the path of thermal_detail.csv or renewables_detail.csv in the prescient results.
        gen_name: str, the name of the generator.
        gen_type: str, fossil or renew, the result from Prescient varies from generator types.
        other_info: list, ["Curtailment"], the information you want to read, such as Curtailment, Unit Cost
    """, 
    gdf = _prescient_output_to_df(gen_details_path)
    if gen_type == "fossil":
        info_list = ["Datetime","Dispatch","Dispatch DA"]
    if gen_type == "renew":
        info_list = ["Datetime","Output","Output DA"]
    # add other information you want to besides the default DA/RT dispatch
    if other_info is not None: 
        for info in other_info:
            info_list.append(info)
    gdf = gdf[gdf["Generator"] == gen_name][info_list]
    gdf.set_index("Datetime", inplace=True)
    
    # rename the columns by adding the generator name.
    new_col_name = {}
    for i in info_list:
        new_col_name[i] = f"{gen_name}_{i}"
    
    if dispatch_path == None:
        print("Empty path, make a new df")
        gdf = gdf.rename(columns=new_col_name)
        dispatch_df = gdf
        dispatch_df.to_csv("Generator_Dispatch.csv")
    else:
        print(f"Extracting dispatch for {gen_name}")
        gdf = gdf.rename(columns=new_col_name)
        dispatch_df = pd.read_csv(dispatch_path).set_index("Datetime")
        # check if the bus has already been read
        for j in info_list:
            if f'{gen_name}_{j}' in dispatch_df.columns:
                print(f"{gen_name}_{j} already exists.")
            else:
                # merge the previous and current one
                gdf_aligned = gdf.reindex(dispatch_df.index)
                dispatch_df_merge = pd.concat([dispatch_df, gdf_aligned], axis=1)
                dispatch_df_merge.to_csv(dispatch_path)
    
    return

In [ ]:
# Here, iterate over all the fossil generators to get dispatch and LMP
# LMP
for idx, key in enumerate(list(gen_dict["fossil"].keys())):
    if idx == 0:
        make_lmp_csv(lmp_path=None, bus_details_path=bus_detail_path, bus_name=gen_dict["fossil"][key]["bus_name"])
    else:
        make_lmp_csv(lmp_path="Bus_LMP.csv", bus_details_path=bus_detail_path, bus_name=gen_dict["fossil"][key]["bus_name"])

Empty path, make a new df
Extracting LMP for Abel
Abel LMP already exists.
Extracting LMP for Abel
Abel LMP already exists.
Extracting LMP for Abel
Abel LMP already exists.
Extracting LMP for Adams
Extracting LMP for Adams
Adams LMP already exists.
Extracting LMP for Adams
Adams LMP already exists.
Extracting LMP for Adams
Adams LMP already exists.
Extracting LMP for Alder
Extracting LMP for Arne
Extracting LMP for Arne
Arne LMP already exists.
Extracting LMP for Arne
Arne LMP already exists.
Extracting LMP for Arne
Arne LMP already exists.
Extracting LMP for Arthur
Extracting LMP for Arthur
Arthur LMP already exists.
Extracting LMP for Arthur
Arthur LMP already exists.
Extracting LMP for Asser
Extracting LMP for Astor
Extracting LMP for Austen
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Austen
Austen LMP already exists.
Extracting LMP for Bach
Extractin

In [ ]:
# Here, iterate over all the fossil generators to get dispatch and LMP
# Dispatch
other_info = ["Unit Cost", "Unit State"]
for idx, key in enumerate(list(gen_dict["fossil"].keys())):
    if idx == 0:
        make_dispatch_csv(dispatch_path=None, gen_details_path=thermal_detail_path, gen_name=key, gen_type="fossil", other_info=other_info)
    else:
        make_dispatch_csv(dispatch_path="Generator_Dispatch.csv", gen_details_path=thermal_detail_path, gen_name=key, gen_type="fossil", other_info=other_info)

Empty path, make a new df
Extracting dispatch for 101_CT_2
Extracting dispatch for 101_STEAM_3
Extracting dispatch for 101_STEAM_4
Extracting dispatch for 102_CT_1
Extracting dispatch for 102_CT_2
Extracting dispatch for 102_STEAM_3
Extracting dispatch for 102_STEAM_4
Extracting dispatch for 107_CC_1
Extracting dispatch for 113_CT_1
Extracting dispatch for 113_CT_2
Extracting dispatch for 113_CT_3
Extracting dispatch for 113_CT_4
Extracting dispatch for 115_STEAM_1
Extracting dispatch for 115_STEAM_2
Extracting dispatch for 115_STEAM_3
Extracting dispatch for 116_STEAM_1
Extracting dispatch for 118_CC_1
Extracting dispatch for 123_STEAM_2
Extracting dispatch for 123_STEAM_3
Extracting dispatch for 123_CT_1
Extracting dispatch for 123_CT_4
Extracting dispatch for 123_CT_5
Extracting dispatch for 201_CT_1
Extracting dispatch for 201_CT_2
Extracting dispatch for 201_STEAM_3
Extracting dispatch for 202_CT_1
Extracting dispatch for 202_CT_2
Extracting dispatch for 202_STEAM_3
Extracting dis

## Check the Summation

In [ ]:
# You need to replace the path to the file you saved.
dispatch_path = "Generator_Dispatch.csv"
lmp_path = "Bus_LMP.csv"

df_lmp = pd.read_csv(lmp_path)
df_dispatch = pd.read_csv(dispatch_path)

print(df_lmp.columns)
print(df_dispatch.columns)

Index(['Datetime', 'Abel_LMP', 'Abel_LMP DA', 'Adams_LMP', 'Adams_LMP DA',
       'Alder_LMP', 'Alder_LMP DA', 'Arne_LMP', 'Arne_LMP DA', 'Arthur_LMP',
       'Arthur_LMP DA', 'Asser_LMP', 'Asser_LMP DA', 'Astor_LMP',
       'Astor_LMP DA', 'Austen_LMP', 'Austen_LMP DA', 'Bach_LMP',
       'Bach_LMP DA', 'Bacon_LMP', 'Bacon_LMP DA', 'Baker_LMP', 'Baker_LMP DA',
       'Barlow_LMP', 'Barlow_LMP DA', 'Barton_LMP', 'Barton_LMP DA',
       'Basov_LMP', 'Basov_LMP DA', 'Bayle_LMP', 'Bayle_LMP DA', 'Behring_LMP',
       'Behring_LMP DA', 'Bloch_LMP', 'Bloch_LMP DA', 'Cabell_LMP',
       'Cabell_LMP DA', 'Cabot_LMP', 'Cabot_LMP DA', 'Carew_LMP',
       'Carew_LMP DA', 'Cecil_LMP', 'Cecil_LMP DA', 'Chase_LMP',
       'Chase_LMP DA', 'Chifa_LMP', 'Chifa_LMP DA', 'Clark_LMP',
       'Clark_LMP DA', 'Cobb_LMP', 'Cobb_LMP DA', 'Cole_LMP', 'Cole_LMP DA',
       'Comte_LMP', 'Comte_LMP DA'],
      dtype='object')
Index(['Datetime', '101_CT_1_Dispatch', '101_CT_1_Dispatch DA',
       '101_CT_1_Unit C

### Calculate the total dispatch

In [ ]:
# calculate the total dispatch
gen_names = list(gen_dict["fossil"].keys())
dispatch_result = {}

for name in gen_names:
    # calculate annual DA/RT dispatch
    dispatch_result[name] = {}
    dispatch_result[name]["tot_Dispatch_DA"] = df_dispatch[name+"_Dispatch DA"].sum()
    dispatch_result[name]["tot_Dispatch"] = df_dispatch[name+"_Dispatch"].sum()
    
# dispatch_result

### Summarize the LMP infomation

In [ ]:
LMP_result = {}

for name in gen_names:
    bus_name = gen_dict["fossil"][name]["bus_name"]
    LMP_result[bus_name] = {}
    LMP_result[bus_name][f"LMP_DA_mean"] = df_lmp[f"{bus_name}_LMP DA"].mean()
    LMP_result[bus_name][f"LMP_DA_median"] = df_lmp[f"{bus_name}_LMP DA"].median()
    LMP_result[bus_name][f"LMP_DA_min"] = df_lmp[f"{bus_name}_LMP DA"].min()
    LMP_result[bus_name][f"LMP_DA_max"] = df_lmp[f"{bus_name}_LMP DA"].max()
    LMP_result[bus_name][f"LMP_mean"] = df_lmp[f"{bus_name}_LMP"].mean()
    LMP_result[bus_name][f"LMP_median"] = df_lmp[f"{bus_name}_LMP"].median()
    LMP_result[bus_name][f"LMP_min"] = df_lmp[f"{bus_name}_LMP"].min()
    LMP_result[bus_name][f"LMP_max"] = df_lmp[f"{bus_name}_LMP"].max()
    
# LMP_result

In [ ]:
# put them in one json file and save.
result_summary = {}
result_summary["LMP"] = LMP_result
result_summary["Dispatch"] = dispatch_result

# change the path before run the cell if needed
PCM_result_path = "PCM_result.json"
with open(PCM_result_path, "w") as f:
    json.dump(result_summary, f)

# Renewable

In [ ]:
# Here, iterate over all the renewable generators to get dispatch and LMP
# LMP
for idx, key in enumerate(list(gen_dict["renew"].keys())):
    make_lmp_csv(lmp_path="Bus_LMP.csv", bus_details_path=bus_detail_path, bus_name=gen_dict["renew"][key]["bus_name"])

Extracting LMP for Aubrey
Extracting LMP for Aubrey
Aubrey LMP already exists.
Extracting LMP for Aubrey
Aubrey LMP already exists.
Extracting LMP for Aubrey
Aubrey LMP already exists.
Extracting LMP for Aubrey
Aubrey LMP already exists.
Extracting LMP for Aubrey
Aubrey LMP already exists.
Extracting LMP for Barton
Barton LMP already exists.
Extracting LMP for Barton
Barton LMP already exists.
Extracting LMP for Barton
Barton LMP already exists.
Extracting LMP for Bell
Extracting LMP for Bell
Bell LMP already exists.
Extracting LMP for Bell
Bell LMP already exists.
Extracting LMP for Bell
Bell LMP already exists.
Extracting LMP for Bell
Bell LMP already exists.
Extracting LMP for Bell
Bell LMP already exists.
Extracting LMP for Cole
Cole LMP already exists.
Extracting LMP for Cole
Cole LMP already exists.
Extracting LMP for Cole
Cole LMP already exists.
Extracting LMP for Cole
Cole LMP already exists.
Extracting LMP for Clive
Extracting LMP for Chain
Extracting LMP for Chain
Chain LMP 

In [ ]:
other_info = ["Curtailment"]
for idx, key in enumerate(list(gen_dict["renew"].keys())):
        make_dispatch_csv(dispatch_path="Generator_Dispatch.csv", gen_details_path=renew_detail_path, gen_name=key, gen_type="renew", other_info=other_info)

Extracting dispatch for 122_HYDRO_1
Extracting dispatch for 122_HYDRO_2
Extracting dispatch for 122_HYDRO_3
Extracting dispatch for 122_HYDRO_4
Extracting dispatch for 122_HYDRO_5
Extracting dispatch for 122_HYDRO_6
Extracting dispatch for 215_HYDRO_1
Extracting dispatch for 215_HYDRO_2
Extracting dispatch for 215_HYDRO_3
Extracting dispatch for 222_HYDRO_1
Extracting dispatch for 222_HYDRO_2
Extracting dispatch for 222_HYDRO_3
Extracting dispatch for 222_HYDRO_4
Extracting dispatch for 222_HYDRO_5
Extracting dispatch for 222_HYDRO_6
Extracting dispatch for 322_HYDRO_1
Extracting dispatch for 322_HYDRO_2
Extracting dispatch for 322_HYDRO_3
Extracting dispatch for 322_HYDRO_4
Extracting dispatch for 320_PV_1
Extracting dispatch for 314_PV_1
Extracting dispatch for 314_PV_2
Extracting dispatch for 313_PV_1
Extracting dispatch for 314_PV_3
Extracting dispatch for 314_PV_4
Extracting dispatch for 313_PV_2
Extracting dispatch for 310_PV_1
Extracting dispatch for 324_PV_1
Extracting dispatch

## Check Benchmark Results

Xinhe and Kay run PCM simulations in their own environment. Use the following codes to make sure they have the consistent outputs from PCM simulations.

### Specify the input paths and read them.

In [ ]:
xc_path = "PCM_result_XC_new.json"
kay_path = "PCM_result_Kay.json"

with open(xc_path, "rb") as f:
    xc_result = json.load(f)

with open(kay_path, "rb") as f:
    kay_result = json.load(f)

### Check LMP

In [ ]:
for k1, k2 in zip(xc_result["LMP"].keys(), kay_result["LMP"].keys()):
#     print(xc_result["LMP"][k1]['LMP_DA_mean'] - kay_result["LMP"][k2]['LMP_DA_mean'])
#     print(xc_result["LMP"][k1]['LMP_DA_median'] - kay_result["LMP"][k2]['LMP_DA_median'])
#     print(xc_result["LMP"][k1]['LMP_DA_min'] - kay_result["LMP"][k2]['LMP_DA_min'])
#     print(xc_result["LMP"][k1]['LMP_DA_max'] - kay_result["LMP"][k2]['LMP_DA_max'])
#     print(xc_result["LMP"][k1]['LMP_mean'] - kay_result["LMP"][k2]['LMP_mean'])
#     print(xc_result["LMP"][k1]['LMP_median'] - kay_result["LMP"][k2]['LMP_median'])
#     print(xc_result["LMP"][k1]['LMP_min'] - kay_result["LMP"][k2]['LMP_min'])
    print(xc_result["LMP"][k1]['LMP_max'] - kay_result["LMP"][k2]['LMP_max'])

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


### Check Dispatch

In [ ]:
for k1, k2 in zip(xc_result["Dispatch"].keys(), kay_result["Dispatch"].keys()):
    print(xc_result["Dispatch"][k1]["tot_Dispatch_DA"]  - kay_result["Dispatch"][k2]["tot_Dispatch_DA"])
    print(xc_result["Dispatch"][k1]["tot_Dispatch"]  - kay_result["Dispatch"][k2]["tot_Dispatch"])

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
